# Notebook 2: Exploring Protein Secondary Structure

## Bio 4525: Structural Bioinformatics of Proteins
**Washington University in St. Louis**

---

## What is Secondary Structure?

While primary structure is the sequence of amino acids (the 1D chain), **secondary structure** describes the local, regular 3D folding patterns that form spontaneously as the chain collapses. These patterns are stabilized by **hydrogen bonds** between atoms in the polypeptide backbone — specifically between the N–H of one residue and the C=O of another.

### Alpha Helix (α-helix)

The alpha helix is a right-handed spiral in which the backbone coils around a central axis. Its key features are:

- **Hydrogen bonds** form between the backbone N–H of residue *i* and the C=O of residue *i+4* (four residues earlier along the chain)
- There are **3.6 amino acids per turn**, with a rise of 1.5 Å per residue along the helix axis
- The **side chains point outward** from the helix, away from the central axis
- In cartoon representations, helices appear as **coiled ribbons or cylinders**

Alpha helices are common in membrane-spanning proteins, in globin proteins like hemoglobin and myoglobin, and in coiled-coil structural proteins like keratin.

### Beta Sheet (β-sheet)

Beta sheets form when two or more extended polypeptide strands (beta strands) line up side by side, connected by backbone hydrogen bonds. The strands can be arranged in two ways:

- **Parallel beta sheet:** All strands run in the same direction (N→C). The hydrogen bonds are oblique (slanted). Parallel sheets tend to be slightly less stable than antiparallel.
- **Antiparallel beta sheet:** Adjacent strands run in opposite directions (N→C then C→N alternating). The hydrogen bonds are perpendicular to the strands and are stronger. Antiparallel sheets are very common in proteins.

In both cases, the side chains alternate above and below the plane of the sheet. In cartoon representations, strands appear as **flat arrows** pointing from N to C terminus.

### Loop / Coil

Loops (also called coils or turns) are irregular segments that connect helices and strands. They lack a regular repeating pattern of hydrogen bonds. Despite appearing "disordered," loops are often the most functionally important parts of a protein — they form active sites, binding interfaces, and allosteric sites. Shorter loops that make a sharp turn while forming one H-bond are called **turns**.

---

## Our Protein of Focus: Ribonuclease A (RNase A)

In this notebook we use **bovine pancreatic ribonuclease A** (PDB: **8RAT**) as our primary example. RNase A is an excellent teaching protein because it contains all three types of secondary structure — helices, sheets, and loops — making its composition much more informative than a purely helical protein. RNase A is also historically significant: it was one of the first enzymes to have its 3D structure determined, and it was used in Anfinsen's Nobel Prize-winning experiments showing that protein sequence alone determines 3D structure.

## Learning Objectives

By the end of this notebook, you will be able to:

- Download a protein structure file (PDB format) programmatically using BioPython
- Navigate BioPython's SMCRA hierarchy to extract structural information
- Use DSSP to assign secondary structure to each residue
- Visualize secondary structure composition with pie charts and bar charts
- Display an interactive 3D structure using nglview
- Map secondary structure elements along the protein sequence
- Interpret a Ramachandran plot of backbone dihedral angles

**Estimated time:** 60–75 minutes

In [ ]:
# -----------------------------------------------------------------------
# SETUP: Install and import all required libraries
# Run this cell first — it only needs to run once per Colab session
#
# IMPORTANT: After this cell finishes installing, go to:
#   Runtime → Restart session
# Then run ALL cells again from the top.
# This restart is required for nglview's 3D viewer to display correctly.
# -----------------------------------------------------------------------

# Install BioPython and nglview
# We pin nglview to version 3.0.8 — the most stable version for Colab
!pip install biopython nglview==3.0.8 --quiet

# -----------------------------------------------------------------------
# Special setup for nglview in Google Colab
# This MUST be called before importing nglview, and AFTER a runtime restart.
# It enables Colab to display interactive 3D widgets inside notebook cells.
# -----------------------------------------------------------------------
from google.colab import output
output.enable_custom_widget_manager()

# -----------------------------------------------------------------------
# Import statements
# -----------------------------------------------------------------------
import os                          # Built-in: file and path operations
import shutil                      # Built-in: check if programs are installed
import pandas as pd                # Pandas: organizes data into tables
import matplotlib.pyplot as plt    # Matplotlib: creates charts and graphs
import numpy as np                 # NumPy: numeric arrays for bar chart layout
import nglview as nv               # nglview: interactive 3D molecular visualization

from Bio.PDB import PDBList, PDBParser    # BioPython: download and parse PDB files
from Bio.PDB.DSSP import DSSP            # BioPython: secondary structure assignment

print('All libraries imported successfully!')
print(f'nglview version: {nv.__version__}')

---
## Section 1: Downloading a PDB Structure

### What is the Protein Data Bank (PDB)?

The **Protein Data Bank (PDB)** is the worldwide archive of experimentally determined protein structures. Each structure is assigned a unique 4-character **PDB ID** (e.g., `8RAT`). The structures were determined by techniques such as X-ray crystallography, NMR spectroscopy, or cryo-electron microscopy, and the coordinates of every atom are deposited as a **PDB file** — a plain text file describing where each atom sits in 3D space.

We will work with **PDB entry 8RAT**: a high-resolution crystal structure of bovine pancreatic ribonuclease A (RNase A). RNase A is a 124-residue enzyme that cleaves RNA molecules. It is an ideal teaching protein because it contains all three types of secondary structure — alpha helices, beta sheets, and loops — and is historically significant as one of the first enzymes to have its 3D structure solved and as the subject of Anfinsen's Nobel Prize-winning protein folding experiments.

### BioPython's SMCRA Hierarchy

BioPython organizes a PDB structure into a nested hierarchy called **SMCRA**. Each level contains the level below it:

```
Structure  (the whole PDB entry, e.g., '8RAT')
└── Model 0  (X-ray structures have just one model)
    └── Chain A  (the single polypeptide chain — 124 residues)
        ├── Residue 1  (LYS)
        │   ├── Atom N
        │   ├── Atom CA  (alpha carbon)
        │   ├── Atom C
        │   ├── Atom O
        │   └── Atom CB
        ├── Residue 2  (GLU)
        └── ... (124 residues total)
```

Understanding this hierarchy is key — we will navigate it to extract chain, residue, and atom information throughout this notebook.

> **A note on single-chain proteins:** Unlike hemoglobin (which has multiple subunits), RNase A is a **monomer** — a single polypeptide chain that folds independently into a functional enzyme. Most proteins are monomers. This makes RNase A a simpler starting point for studying structure.

In [ ]:
# -----------------------------------------------------------------------
# Download the RNase A crystal structure from the Protein Data Bank
# -----------------------------------------------------------------------

# PDBList connects to the RCSB PDB server and downloads the structure file
pdbl = PDBList()

# retrieve_pdb_file() downloads the structure and returns the file path
# file_format='pdb' requests the legacy PDB format (required for DSSP later)
# pdir='.' saves the file in the current directory
# The try/except block catches network errors and gives a clear message
try:
    downloaded_path = pdbl.retrieve_pdb_file(
        '8RAT',
        file_format='pdb',   # Note: newer BioPython uses 'file_format', not 'file_type'
        pdir='.'
    )
except Exception as download_error:
    raise RuntimeError(
        'Could not download 8RAT from RCSB PDB.\n'
        'Check your internet connection and try again.\n'
        f'Original error: {download_error}'
    )

print(f'Downloaded file path: {downloaded_path}')

# PDBList places files in a subdirectory (e.g., ./ra/pdb8rat.ent).
# If the returned path does not exist, search for the file.
if not os.path.exists(downloaded_path):
    print('File not at expected path, searching...')
    for root, dirs, files in os.walk('.'):
        for fname in files:
            if '8rat' in fname.lower():
                downloaded_path = os.path.join(root, fname)
                print(f'Found at: {downloaded_path}')
                break

pdb_filename = downloaded_path

# Parse the PDB file into a BioPython structure object
# QUIET=True suppresses harmless warning messages about non-standard residues
parser = PDBParser(QUIET=True)
structure = parser.get_structure('8RAT', pdb_filename)

print()
print('Structure parsed successfully!')
print(f'Structure ID: {structure.id}')


In [ ]:
# -----------------------------------------------------------------------
# Explore the SMCRA hierarchy of the RNase A structure
# -----------------------------------------------------------------------

# --- Level 1: Models ---
# X-ray crystal structures typically have just one model (Model 0).
# NMR structures may have multiple models (different conformations).
model_list = list(structure.get_models())
print(f'Number of models in 8RAT: {len(model_list)}')
print()

# Select Model 0 (the only model in this X-ray structure)
model = structure[0]

# --- Level 2: Chains ---
chain_list = list(model.get_chains())
print(f'Number of chains: {len(chain_list)}')
print('(RNase A is a monomer — a single polypeptide chain)')
print()

# --- Level 3: Residues ---
# PDB files contain both ATOM records (amino acids) and HETATM records
# (ligands, water molecules, etc.). We filter to get only standard amino acids,
# which have residue ID[0] == ' ' (a space character).
print('Chain breakdown:')
print('-' * 40)
for chain in chain_list:
    amino_acid_residues = [r for r in chain.get_residues() if r.id[0] == ' ']
    print(f'  Chain {chain.id}: {len(amino_acid_residues)} amino acid residues')

print()
total_residues = sum(
    len([r for r in chain.get_residues() if r.id[0] == ' '])
    for chain in chain_list
)
print(f'Total amino acid residues: {total_residues}')
print()
print('For comparison: hemoglobin (1HHO) has 2 chains in its asymmetric unit (141 + 146 = 287 residues).')
print('RNase A (8RAT) is a much smaller protein, which makes it a great model system.')

---
## Section 2: Identifying Secondary Structure with DSSP

### What is DSSP?

**DSSP** (Dictionary of Secondary Structure of Proteins) is a standard algorithm developed by Kabsch and Sander (1983) that reads the 3D atomic coordinates of a protein and assigns each residue a secondary structure label based on hydrogen bond geometry and backbone angles. It is the most widely used method for secondary structure assignment from crystal structures.

DSSP uses a single-letter code for each residue:

| Code | Structure | We classify it as |
|------|-----------|------------------|
| `H` | Alpha helix | Helix |
| `G` | 3₁₀ helix (tighter than alpha) | Helix |
| `I` | Pi helix (wider than alpha) | Helix |
| `E` | Extended beta strand | Sheet |
| `B` | Isolated beta bridge | Sheet |
| `T` | Hydrogen-bonded turn | Loop/Other |
| `S` | Bend | Loop/Other |
| `-` | Coil (no regular structure) | Loop/Other |

DSSP is an external program — not a Python package — so we need to install it separately before BioPython can call it. The next cell handles that installation.

In [ ]:
# -----------------------------------------------------------------------
# Install the DSSP program
# DSSP is a standalone binary that BioPython calls behind the scenes.
# The apt-get command below installs it from Ubuntu's package repository.
# -----------------------------------------------------------------------
print('Installing DSSP...')
!apt-get install dssp -y -q
print()

# After installation, the binary may be called 'dssp' or 'mkdssp'
# depending on the version. We check which one is available.
if shutil.which('mkdssp'):
    dssp_binary = 'mkdssp'
elif shutil.which('dssp'):
    dssp_binary = 'dssp'
else:
    print('WARNING: DSSP binary not found after installation.')
    print('Try running this cell again, or restart the runtime and re-run from the top.')
    dssp_binary = 'mkdssp'  # Default — change to dssp if this fails

print(f'DSSP binary to use: {dssp_binary}')

In [ ]:
# -----------------------------------------------------------------------
# Run DSSP and build a DataFrame of secondary structure assignments
# -----------------------------------------------------------------------

print('Running DSSP secondary structure assignment...')
print('(This reads the 3D coordinates and assigns a code to each residue)')
print()

try:
    # DSSP() takes the model object, the PDB file path, the binary name,
    # and file_type='PDB' to tell it we are using the legacy PDB format.
    dssp_result = DSSP(model, pdb_filename, dssp=dssp_binary, file_type='PDB')
    print(f'DSSP completed. Secondary structure assigned to {len(dssp_result)} residues.')

except Exception as error:
    print(f'DSSP error: {error}')
    print('Try re-running the installation cell (cell above) and then this cell again.')
    raise

print()

# -----------------------------------------------------------------------
# Build a pandas DataFrame from the DSSP results
# Each row = one residue, with its chain, residue number, and SS code
# -----------------------------------------------------------------------
ss_data = []

# dssp_result is a dictionary-like object.
# Each key is a tuple: (chain_id, residue_id_tuple)
# Each value is a tuple of DSSP outputs for that residue.
for key_tuple in dssp_result.keys():
    chain_id = key_tuple[0]              # Chain identifier: 'A' or 'B'
    residue_id_tuple = key_tuple[1]      # Residue ID: (hetflag, seq_number, insertion_code)
    residue_number = residue_id_tuple[1] # The numeric sequence number

    dssp_output = dssp_result[key_tuple]
    amino_acid = dssp_output[1]          # Single-letter amino acid code
    ss_code = dssp_output[2]             # DSSP secondary structure code

    ss_data.append([residue_number, amino_acid, chain_id, ss_code])

# Create the DataFrame
ss_df = pd.DataFrame(ss_data, columns=['Residue_Number', 'Amino_Acid', 'Chain', 'SS_Code'])

print('First 15 rows of the secondary structure DataFrame:')
print('=' * 55)
print(ss_df.head(15).to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# Classify each residue into Helix, Sheet, or Loop/Other
# -----------------------------------------------------------------------

def classify_ss(ss_code):
    '''
    Map a DSSP single-letter code to a broad secondary structure category.

    Parameters:
    -----------
    ss_code : str
        A single DSSP code character (H, G, I, E, B, T, S, or -)

    Returns:
    --------
    str: one of 'Helix', 'Sheet', or 'Loop/Other'
    '''
    if ss_code in ('H', 'G', 'I'):
        return 'Helix'
    elif ss_code in ('E', 'B'):
        return 'Sheet'
    else:
        return 'Loop/Other'


# Apply the function to every row in the DataFrame
# .apply() runs the function once per row and stores the result in a new column
ss_df['SS_Category'] = ss_df['SS_Code'].apply(classify_ss)

# Count how many residues fall into each category
category_counts = ss_df['SS_Category'].value_counts()

print('Secondary Structure Composition of RNase A (8RAT):')
print('=' * 50)
for category in ['Helix', 'Sheet', 'Loop/Other']:
    count = category_counts.get(category, 0)
    percentage = (count / len(ss_df)) * 100
    print(f'  {category:<12}: {count:>4} residues  ({percentage:.1f}%)')

print()
print(f'Total residues with SS assignment: {len(ss_df)}')
print()
print('RNase A contains all three secondary structure types —')
print('a mixed alpha/beta protein, in contrast to the all-helical hemoglobin.')

In [ ]:
# -----------------------------------------------------------------------
# Pie chart: secondary structure proportions in RNase A
# -----------------------------------------------------------------------

# Define categories and matching colors
categories = ['Helix', 'Sheet', 'Loop/Other']
counts = [category_counts.get(cat, 0) for cat in categories]
colors = ['#4472C4', '#ED7D31', '#A9D18E']  # Blue, orange, green

fig, ax = plt.subplots(figsize=(7, 7))

# Create the pie chart
# autopct shows the percentage label inside each slice
# startangle=90 places the first slice at the 12 o'clock position
wedges, texts, autotexts = ax.pie(
    counts,
    labels=categories,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

# Make the percentage labels larger and bold
for autotext in autotexts:
    autotext.set_fontsize(13)
    autotext.set_fontweight('bold')

ax.set_title('Secondary Structure Composition of RNase A (8RAT)',
             fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print()
print('Unlike hemoglobin (which is ~75% helical), RNase A is a mixed alpha/beta protein.')
print('The presence of all three secondary structure types makes it a great model for')
print('understanding how helices, sheets, and loops work together in a folded protein.')

### 💭 Think About It

Look at the pie chart and composition summary you just generated for RNase A.

1. **What fraction of RNase A is helical? What fraction is in beta strands?** Compare this to what you know about hemoglobin (~75% helix). Proteins with a mix of helices and sheets are sometimes called **alpha/beta proteins** — they are very common in nature.

2. **What is the role of the loop/coil regions in RNase A?** RNase A has two histidine residues (His12 and His119) in its active site that are positioned by surrounding loops. Why might a flexible loop be a better location for an active-site residue than a rigid helix?

3. **The DSSP algorithm assigns structure based on 3D coordinates, not sequence.** What does this tell you about the relationship between primary structure and secondary structure? Does the amino acid sequence alone determine what secondary structure will form?

4. **Think about the concept of structural diversity:** Some proteins are almost entirely helical (hemoglobin), some are almost entirely beta-sheet (antibody domains), and some are mixed (RNase A). Can you think of a biological reason why proteins with different functions might have different secondary structure compositions?

---
## Section 3: Visualizing the 3D Structure with nglview

Numbers and charts tell us the *composition* of a protein, but to truly understand secondary structure we need to **see** it in 3D. We will use **nglview**, an interactive molecular viewer designed to run inside Jupyter notebooks and Google Colab.

### Color Scheme: Secondary Structure (`sstruc`)

We will color the structure using the `sstruc` scheme, which assigns a color to each DSSP secondary structure code. Because DSSP distinguishes several sub-types of helix and loop, the result is more colorful than a simple three-color scheme — this is normal and expected:

| Color | DSSP Code | Structure |
|-------|-----------|-----------|
| **Red** | H | Alpha helix |
| **Deep pink** | G | 3₁₀ helix (tighter than alpha) |
| **Blue/purple** | I | Pi helix (wider than alpha) |
| **Yellow** | E | Beta strand |
| **Green** | T | Hydrogen-bonded turn |
| **Cyan** | S | Bend |
| **White** | – | Coil (no regular structure) |

The overall appearance is a **rainbow-like ribbon** — this is what `sstruc` is supposed to look like. You can use the colors to identify secondary structure type for any segment.

The visualization uses a **cartoon** representation — helices appear as coiled ribbons, strands appear as flat arrows pointing from N to C terminus, and loops/turns appear as thin tubes.

### How to Interact with the 3D Viewer

| Action | Effect |
|---|---|
| **Left-click + drag** | Rotate the structure |
| **Right-click + drag** | Pan / translate the view |
| **Scroll wheel** | Zoom in or out |
| **Double-click** | Center the view on a residue |
| **Ctrl + click** | Pick an atom and display its identity |

> **Note:** The 3D viewer may take a few seconds to load. If it appears blank, try scrolling up and down in the notebook, or re-run this cell. Make sure you restarted the Colab runtime after the setup cell installed nglview.

In [ ]:
# -----------------------------------------------------------------------
# Display the RNase A structure in 3D using nglview
#
# If the viewer appears blank or does not load:
#   1. Make sure you restarted the runtime after the setup cell installed nglview
#   2. Re-run ALL cells from the top in order
#   3. Wait a few seconds — the viewer can take time to initialize
# -----------------------------------------------------------------------

# By passing 'representations' at initialization, we skip the default
# N→C rainbow representation entirely and load only the sstruc coloring.
# colorScheme='sstruc' color mapping:
#   Red        → Alpha helix (H)
#   Deep pink  → 3₁₀ helix (G)
#   Yellow     → Beta strand (E)
#   Green      → Turn (T)
#   Cyan       → Bend (S)
#   White      → Coil (-)
view = nv.show_structure_file(
    pdb_filename,
    representations=[{
        'type': 'cartoon',
        'params': {'colorScheme': 'sstruc'}
    }]
)

# White background for better contrast
view.background = 'white'

# -----------------------------------------------------------------------
# HOW TO INTERACT WITH THE 3D VIEWER:
#   Left-click + drag    -> rotate the structure
#   Right-click + drag   -> pan / translate the view
#   Scroll wheel         -> zoom in or out
#   Double-click         -> center the view on a residue
#   Ctrl + click         -> pick an atom and display its identity
# -----------------------------------------------------------------------

view  # Must be the last line in the cell to display in Colab

### 💭 Think About It

Rotate and explore the 3D structure of RNase A above.

1. **Can you identify the helices and strands?** Helices appear as coiled ribbons (red/orange), while strands appear as flat arrows (yellow). Loops are the thin connecting segments (white/gray). Are the helices and sheets clustered in particular regions of the protein, or are they interleaved?

2. **RNase A has a kidney-shaped or crescent structure.** Can you see a cleft or groove on the surface? This is the active site — where the enzyme grabs and cleaves RNA. The two active-site histidines (His12 and His119) sit in loops near this cleft. Can you see where the loops are positioned?

3. **Compare the cartoon representation to the schematic diagrams** you generated in the previous section. In the 3D view, do the beta strand arrows point in consistent directions, or alternating directions? This gives you a clue about whether the sheet is parallel or antiparallel.

4. **RNase A was a landmark protein:** Christian Anfinsen used RNase A to prove that the amino acid sequence alone determines the 3D fold (no cellular machinery needed). He denatured RNase A with urea + beta-mercaptoethanol, then removed the denaturant and showed that the enzyme refolded spontaneously. Looking at this structure, what features might drive spontaneous refolding?

---
## Section 5: The Ramachandran Plot

### What is a Ramachandran Plot?

The **Ramachandran plot** is one of the most important tools in structural bioinformatics. It was introduced by G.N. Ramachandran in 1963 and plots two backbone dihedral angles — **phi (φ)** and **psi (ψ)** — for every residue in a protein.

### Understanding Phi and Psi

Each amino acid in the polypeptide backbone has two rotatable bonds:

- **φ (phi):** the bond between the nitrogen (N) and the alpha carbon (Cα). Phi is defined by the atoms C–N–Cα–C.
- **ψ (psi):** the bond between the alpha carbon (Cα) and the carbonyl carbon (C). Psi is defined by the atoms N–Cα–C–N.

Both angles are measured in degrees and range from −180° to +180°. The combination of φ and ψ determines the local backbone geometry — in other words, which secondary structure a residue adopts.

### Favored Regions

Steric clashes between atoms limit which φ/ψ combinations are physically allowed:

| Region | φ (approx.) | ψ (approx.) | Secondary Structure |
|--------|-------------|-------------|---------------------|
| α-helix cluster | −57° | −47° | Alpha helix |
| β-strand cluster | −120° | +130° | Beta sheet |
| Left-handed helix | +57° | +47° | Rare; glycine only |

Residues outside the favored regions experience steric clash — so in a well-resolved crystal structure, almost all residues should cluster in the allowed regions. A Ramachandran plot is therefore also used as a **quality check**: if many residues fall in disallowed regions, the structure may have errors.

> **Glycine is an exception:** Because glycine has only a hydrogen atom as its side chain, it has no steric constraints and can adopt any φ/ψ combination. Glycine residues often appear as outliers in Ramachandran plots.

In [ ]:
# -----------------------------------------------------------------------
# Extract phi (φ) and psi (ψ) backbone dihedral angles from DSSP results
#
# The DSSP output tuple for each residue contains:
#   Index [0] : DSSP sequential number
#   Index [1] : amino acid (single letter)
#   Index [2] : secondary structure code (H, E, -, etc.)
#   Index [3] : relative solvent accessibility
#   Index [4] : phi angle (degrees)
#   Index [5] : psi angle (degrees)
#   (more indices follow but we don't need them here)
#
# DSSP uses 360.0 as a sentinel value for "undefined angle".
# The first and last residues of a chain cannot have both phi and psi
# defined, so we skip any residue where either angle = 360.0.
# -----------------------------------------------------------------------

phi_psi_data = []

for key_tuple in dssp_result.keys():
    chain_id        = key_tuple[0]
    residue_id_tuple = key_tuple[1]
    residue_number  = residue_id_tuple[1]

    dssp_output = dssp_result[key_tuple]
    amino_acid  = dssp_output[1]
    ss_code     = dssp_output[2]
    phi_angle   = dssp_output[4]    # phi in degrees
    psi_angle   = dssp_output[5]    # psi in degrees

    # Skip residues where either angle is undefined (DSSP uses 360.0)
    if phi_angle == 360.0 or psi_angle == 360.0:
        continue

    ss_category = classify_ss(ss_code)

    phi_psi_data.append([residue_number, amino_acid, chain_id,
                          ss_code, ss_category, phi_angle, psi_angle])


# Build a DataFrame from the extracted angles
rama_df = pd.DataFrame(
    phi_psi_data,
    columns=['Residue_Number', 'Amino_Acid', 'Chain',
             'SS_Code', 'SS_Category', 'Phi', 'Psi']
)

print(f'Residues with valid phi/psi angles: {len(rama_df)}')
print(f'(Residues at chain termini have no phi or psi — these are excluded)')
print()
print('First 12 rows:')
print(rama_df.head(12).to_string(index=False))
print()

# Print the phi/psi summary by category
print('Mean phi and psi angles by secondary structure category:')
print('=' * 55)
for cat in ['Helix', 'Sheet', 'Loop/Other']:
    subset = rama_df[rama_df['SS_Category'] == cat]
    if len(subset) > 0:
        mean_phi = subset['Phi'].mean()
        mean_psi = subset['Psi'].mean()
        print(f'  {cat:<12}  n={len(subset):>3}   φ_mean={mean_phi:>7.1f}°   ψ_mean={mean_psi:>7.1f}°')

In [ ]:
# -----------------------------------------------------------------------
# Ramachandran plot: phi (φ) vs psi (ψ), colored by secondary structure
#
# Each point = one residue in RNase A.
# Residues cluster in predictable regions based on their SS type.
# -----------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 8))

# Color scheme matches the pie chart and sequence map
rama_colors = {
    'Helix':      '#4472C4',   # blue
    'Sheet':      '#ED7D31',   # orange
    'Loop/Other': '#70AD47'    # green
}

# Plot each SS category as a separate scatter series so it appears in the legend
for category in ['Helix', 'Sheet', 'Loop/Other']:
    subset = rama_df[rama_df['SS_Category'] == category]
    ax.scatter(
        subset['Phi'],
        subset['Psi'],
        c=rama_colors[category],
        label=f'{category}  (n={len(subset)})',
        alpha=0.75,
        s=55,
        edgecolors='white',
        linewidths=0.4,
        zorder=3
    )

# -----------------------------------------------------------------------
# Draw reference lines at 0° to divide the plot into quadrants
# -----------------------------------------------------------------------
ax.axhline(y=0, color='gray', linewidth=0.8, linestyle='-', alpha=0.4, zorder=1)
ax.axvline(x=0, color='gray', linewidth=0.8, linestyle='-', alpha=0.4, zorder=1)

# -----------------------------------------------------------------------
# Mark the expected cluster centers for alpha helix and beta strand
# These dashed circles are guides — not hard boundaries
# -----------------------------------------------------------------------
# Alpha helix: φ ≈ −57°, ψ ≈ −47°
helix_circle = plt.Circle((-57, -47), radius=25, fill=False,
                            color='#1565C0', linewidth=2,
                            linestyle='--', alpha=0.7, zorder=2)
ax.add_patch(helix_circle)
ax.text(-57, -47, 'α', fontsize=16, ha='center', va='center',
        color='#1565C0', fontweight='bold')

# Beta strand: φ ≈ −120°, ψ ≈ +130°
strand_circle = plt.Circle((-120, 130), radius=25, fill=False,
                             color='#BF360C', linewidth=2,
                             linestyle='--', alpha=0.7, zorder=2)
ax.add_patch(strand_circle)
ax.text(-120, 130, 'β', fontsize=16, ha='center', va='center',
        color='#BF360C', fontweight='bold')

# -----------------------------------------------------------------------
# Axis labels, title, and formatting
# -----------------------------------------------------------------------
ax.set_xlabel('φ (Phi) angle (degrees)', fontsize=13)
ax.set_ylabel('ψ (Psi) angle (degrees)', fontsize=13)
ax.set_title('Ramachandran Plot of RNase A (8RAT)\nBackbone dihedral angles colored by secondary structure',
             fontsize=13, fontweight='bold')

ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.set_xticks(range(-180, 181, 60))
ax.set_yticks(range(-180, 181, 60))
ax.set_aspect('equal')

ax.legend(fontsize=11, loc='lower right', framealpha=0.9)

# Light grid for readability
ax.grid(True, linestyle='--', alpha=0.25, zorder=0)

plt.tight_layout()
plt.show()

print()
print('Reading the plot:')
print('  • Blue points (Helix) should cluster near φ≈−57°, ψ≈−47° (upper-left α circle)')
print('  • Orange points (Sheet) should cluster near φ≈−120°, ψ≈+130° (upper-left β circle)')
print('  • Green points (Loop) are more scattered — loops can adopt many conformations')
print('  • Points outside the dashed circles are in less common but often still allowed regions')

### 💭 Think About It

Look at the Ramachandran plot you just generated.

1. **Do the helix residues (blue) cluster near φ ≈ −57°, ψ ≈ −47°?** This is where the alpha helix region is expected. How tight is the cluster — are all the helix residues close together, or are some scattered? A tight cluster indicates a well-defined, regular helix.

2. **Do the strand residues (orange) cluster near φ ≈ −120°, ψ ≈ +130°?** Note that the strand region is broader than the helix region — extended beta conformations allow somewhat more variation in angle. Does the RNase A data reflect this?

3. **Where do the loop residues (green) fall?** Loops are not constrained by regular H-bonding patterns, so they can sample a wider range of φ/ψ combinations. Do you see green points scattered throughout the plot, or do they cluster in particular regions?

4. **If you see any points far outside the expected regions**, they could represent:
   - **Glycine residues** (no side chain → no steric constraint → any φ/ψ allowed)
   - **Proline residues** (rigid ring structure → fixed φ ≈ −60°, restricted ψ)
   - **Crystal structure artifacts** (data at the limit of resolution)
   
   The DSSP output includes the amino acid identity — could you modify the Ramachandran code to color glycines differently from other residues?

5. **Ramachandran plots as quality metrics:** Software like MolProbity scores protein structures partly based on how many residues fall in the "favored" vs. "allowed" vs. "disallowed" zones. Why might a structure with many disallowed residues be considered unreliable?

---
## Summary

In this notebook, you:

- **Downloaded** the RNase A crystal structure (PDB: 8RAT) programmatically using BioPython's `PDBList`
- **Navigated** BioPython's SMCRA hierarchy (Structure → Model → Chain → Residue → Atom) to confirm that RNase A is a single-chain monomer with 124 residues
- **Assigned** secondary structure to every residue using DSSP, producing a DataFrame with per-residue annotations
- **Quantified** RNase A's secondary structure composition — a mixed alpha/beta protein with helices, strands, and loops — and compared it conceptually to the all-helical hemoglobin
- **Visualized** the 3D structure using nglview with the `sstruc` color scheme, identifying helices (red), strands (yellow), turns (green), and coils (white)
- **Analyzed** backbone dihedral angles (phi and psi) from DSSP and plotted them in a Ramachandran plot, confirming that helix residues cluster near (−57°, −47°) and strand residues near (−120°, +130°)

### Looking Ahead

In **Notebook 3**, we move to **tertiary structure** — the complete 3D arrangement of a protein chain. We will measure atomic distances to identify the catalytic active site of RNase A, detect disulfide bonds (RNase A has four), and explore how the linear sequence folds into a compact, functional 3D shape.

---
## Try It Yourself: Comparing Secondary Structure Across Protein Families

RNase A (8RAT) is a **mixed alpha/beta protein**. But many proteins are dominated by a single secondary structure type. In this extension, you will repeat the analysis for two additional proteins — one that is almost entirely helical and one that is almost entirely beta-sheet — and compare all three.

---

### Protein 1: An All-Helical Protein — Human Hemoglobin

**PDB ID:** `1HHO` (oxygenated human hemoglobin, 2.1 Å resolution)

Hemoglobin is the oxygen carrier in red blood cells. Each subunit folds into the **globin fold** — a bundle of 8 alpha helices that wrap around a heme group. It has essentially no beta sheet content, making it a classic example of an **all-alpha protein**.

The asymmetric unit of 1HHO contains two chains: Chain A (alpha subunit, 141 residues) and Chain B (beta subunit, 146 residues).

**Guiding questions:**
1. How does the pie chart compare to RNase A? Does the helix fraction dominate as expected?
2. Run DSSP and look at the composition table. What percentage of residues are in beta sheet? Is it close to zero?
3. Generate the Ramachandran plot. Since hemoglobin is almost entirely helical, you should see a very tight cluster near φ ≈ −57°, ψ ≈ −47° with almost nothing in the beta-strand region. Does this match?
4. View the 3D structure with nglview using `colorScheme='sstruc'`. Compare the ribbon appearance to RNase A — can you see the difference between an all-helical protein and a mixed alpha/beta protein?

---

### Protein 2: A Beta-Sheet-Rich Protein — Green Fluorescent Protein (GFP)

**PDB ID:** `1GFL` (wild-type *Aequorea victoria* GFP, 1.9 Å resolution)

GFP is the glowing jellyfish protein widely used as a fluorescent tag in cell biology. It folds into an **11-stranded beta-barrel** — a cylinder of antiparallel beta strands enclosing a central helix that contains the fluorophore. It is a classic example of an **all-beta protein** (with one central helix).

GFP has only one chain, similar to RNase A.

**Guiding questions:**
1. How does the pie chart compare to RNase A and hemoglobin? Is sheet now the dominant category?
2. Run DSSP and look at the composition table. What percentage of residues are in alpha helix? What about beta sheet?
3. Generate the Ramachandran plot. You should see a large cluster in the beta-strand region (φ ≈ −120°, ψ ≈ +130°) and a small cluster in the helix region (from the single central helix). Does this match?
4. View the 3D structure with nglview. Can you see the barrel shape? Can you identify the single central helix inside the barrel?
5. The GFP fluorophore forms spontaneously from three residues (Ser65-Tyr66-Gly67) that sit in a loop at the center of the barrel. What secondary structure does DSSP assign to those residues? Why might a loop be a better location for a chromophore than a helix or strand?

---

### Summary Comparison

After completing both analyses, fill in this comparison table:

| Protein | PDB | Helix % | Sheet % | Loop % | Structural class |
|---------|-----|---------|---------|--------|-----------------|
| RNase A | 8RAT | ? | ? | ? | Mixed α/β |
| Hemoglobin | 1HHO | ? | ? | ? | All-α |
| GFP | 1GFL | ? | ? | ? | All-β |

> **Hint:** You can re-use `classify_ss()` for both proteins. Just change the PDB ID in the download cell and re-run from there.

In [ ]:
# -----------------------------------------------------------------------
# Starter code — repeat the analysis for hemoglobin (1HHO) and GFP (1GFL)
# Follow the same steps as the main notebook for each protein:
#
#   Step 1: Download using pdbl.retrieve_pdb_file('1HHO', file_format='pdb', pdir='.')
#   Step 2: Parse the structure with PDBParser
#   Step 3: Run DSSP and build a DataFrame
#   Step 4: Classify residues and create a pie chart
#   Step 5: Visualize with nglview (colorScheme='sstruc')
#   Step 6: Extract phi/psi angles and draw the Ramachandran plot
#
# Then repeat Steps 1-6 for 1GFL (GFP).
# -----------------------------------------------------------------------

# Your code here:
